# Deploy the Customer H2O Model Online

This notebook generates the scoring function and deploys a packaged native H2O binary or MOJO behind an Azure ML managed online endpoint using the environment from notebook 03. Deployment and traffic promotion are separate: first create a zero-traffic deployment, then run optional golden parity in the target runtime before deciding whether to promote.

## Before you run it

- Confirm the customer model and target environment versions are registered.
- Confirm `.env` still points to the same validated bundle. If you replaced the demo, follow `data/h2o/customer_bundle/README.md` and rerun notebooks 01-03 first.
- Confirm `AZUREML_ONLINE_ENDPOINT_IDENTITY_ID` points to the intended managed identity. Endpoint identity cannot be changed after the endpoint is created.
- Confirm that identity can pull from the workspace container registry and read the workspace storage account.
- Leave `DEPLOY_H2O_ENDPOINT=false` for the first pass.
- Leave `PROMOTE_H2O_TRAFFIC=false` until cloud parity passes.

The endpoint uses Microsoft Entra token authentication. Public network access follows `AZUREML_ONLINE_ENDPOINT_PUBLIC_NETWORK_ACCESS`, which is `disabled` in the workshop template.

**Source:** Adapted from this repository's H2O endpoint notebook and the Azure ML simple managed endpoint example.

In [1]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
from azure.ai.ml import MLClient
from azure.ai.ml.constants import ManagedServiceIdentityType
from azure.ai.ml.entities import (
    CodeConfiguration,
    IdentityConfiguration,
    ManagedIdentityConfiguration,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
)
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = (
    Path(notebook_file).resolve().parent
    if notebook_file
    else Path.cwd().resolve()
)
for candidate in (search_start, *search_start.parents):
    if (candidate / ".env.example").is_file() and (candidate / "outputs").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")

load_dotenv(WORKSHOP_ROOT / ".env", override=True)

def enabled(name: str) -> bool:
    return os.getenv(name, "false").lower() in {"1", "true", "yes"}

sys.path.insert(0, str(WORKSHOP_ROOT / "src/h2o"))
from customer_profiles import load_selected_manifest

print(f"Workshop root: {WORKSHOP_ROOT}")

Workshop root: /home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop


## 1. Load the deployment contract

We read the dual-format manifest and customer asset names from `.env`. Golden fixtures are optional. When provided, this notebook creates a request and checks parity after deployment; when omitted, that check is reported as skipped.

Read it carefully. These values describe what will be deployed; later cells should not quietly substitute a different version.

In [ ]:
BUNDLE_DIR, manifest = load_selected_manifest(WORKSHOP_ROOT)
MODEL_PATH = BUNDLE_DIR / manifest["model_file"]
golden_data = manifest.get("golden_data", {})
GOLDEN_PROVIDED = golden_data.get("provided", False)
RUN_GOLDEN_VALIDATION = golden_data.get("validate", GOLDEN_PROVIDED)
REQUIRE_GOLDEN_VALIDATION = golden_data.get("required", False)
GOLDEN_INPUT_PATH = (
    BUNDLE_DIR / golden_data["input_file"] if GOLDEN_PROVIDED else None
)
GOLDEN_EXPECTED_PATH = (
    BUNDLE_DIR / golden_data["expected_file"] if GOLDEN_PROVIDED else None
)

MODEL_NAME = manifest["model_name"]
ENVIRONMENT_NAME = manifest["environment_name"]
ENDPOINT_NAME = manifest["endpoint_name"]
DEPLOYMENT_NAME = manifest["deployment_name"]
IDENTITY_ID = os.getenv("AZUREML_ONLINE_ENDPOINT_IDENTITY_ID", "").strip()
PUBLIC_NETWORK_ACCESS = os.getenv(
    "AZUREML_ONLINE_ENDPOINT_PUBLIC_NETWORK_ACCESS",
    "disabled",
)
DEPLOY = enabled("DEPLOY_H2O_ENDPOINT")
PROMOTE = enabled("PROMOTE_H2O_TRAFFIC")

display(
    {
        "profile": manifest["profile"],
        "model": f"{MODEL_NAME}@latest",
        "environment": f"{ENVIRONMENT_NAME}@latest",
        "model_format": manifest["model_format"],
        "model_h2o": manifest["h2o_version"],
        "mojo": manifest.get("mojo_version") or "not applicable",
        "runtime_h2o": manifest["runtime_h2o_version"],
        "golden_data": "provided" if GOLDEN_PROVIDED else "not provided",
        "run_golden_validation": RUN_GOLDEN_VALIDATION,
        "golden_validation_required": REQUIRE_GOLDEN_VALIDATION,
        "endpoint": ENDPOINT_NAME,
        "deployment": DEPLOYMENT_NAME,
        "identity": IDENTITY_ID or "system-assigned at endpoint creation",
        "public_network_access": PUBLIC_NETWORK_ACCESS,
    }
)

## 2. Generate the dual-format scoring function

The scoring script is part of this deployment contract, so the notebook writes and syntax-checks it under ignored `outputs/generated`. It loads native H2O binaries with `h2o.load_model()` and imports MOJO ZIPs with `h2o.upload_mojo()`.

In [3]:
import ast
import textwrap

GENERATED_SCORE_DIR = WORKSHOP_ROOT / "outputs/generated/h2o_customer/online"
GENERATED_SCORE_DIR.mkdir(parents=True, exist_ok=True)
(GENERATED_SCORE_DIR / ".amlignore").write_text(
    "__pycache__/\n*.py[cod]\n", encoding="utf-8"
)
SCORE_PATH = GENERATED_SCORE_DIR / "score.py"

ONLINE_SCORE_SOURCE = r'''
import atexit
import hashlib
import json
import logging
import os
import threading
from pathlib import Path

import h2o
import pandas as pd
from azureml_inference_server_http.api.aml_response import AMLResponse

_model = None
_manifest = None
_predict_lock = threading.Lock()


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _find_manifest(model_root):
    matches = list(model_root.rglob("model_manifest.json"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one model_manifest.json, found {len(matches)}")
    return matches[0]


def _predict(frame):
    h2o_frame = None
    prediction_frame = None
    try:
        h2o_frame = h2o.H2OFrame(frame)
        for column in _manifest.get("categorical_features", []):
            h2o_frame[column] = h2o_frame[column].asfactor()
        prediction_frame = _model.predict(h2o_frame)
        values = prediction_frame.as_data_frame()["predict"]
        return [value.item() if hasattr(value, "item") else value for value in values]
    finally:
        if prediction_frame is not None:
            h2o.remove(prediction_frame)
        if h2o_frame is not None:
            h2o.remove(h2o_frame)


def _parse_request(raw_data):
    payload = json.loads(raw_data) if isinstance(raw_data, (str, bytes)) else raw_data
    input_data = payload.get("input_data") if isinstance(payload, dict) else None
    if not isinstance(input_data, dict):
        raise ValueError("Request must contain an input_data object")
    columns = input_data.get("columns")
    rows = input_data.get("data")
    expected_columns = _manifest["features"]
    if columns != expected_columns:
        raise ValueError(f"Expected columns in this order: {expected_columns}")
    if not isinstance(rows, list) or not 1 <= len(rows) <= 100:
        raise ValueError("Request must contain between 1 and 100 rows")
    if any(not isinstance(row, list) or len(row) != len(columns) for row in rows):
        raise ValueError("Every row must have one value for each column")
    frame = pd.DataFrame(rows, columns=columns)
    categorical = set(_manifest.get("categorical_features", []))
    for column in set(expected_columns) - categorical:
        frame[column] = pd.to_numeric(frame[column], errors="raise")
    for column in categorical:
        frame[column] = frame[column].astype("string")
    return frame


def _shutdown_h2o():
    try:
        if h2o.connection() is not None:
            h2o.cluster().shutdown(prompt=False)
    except Exception:
        logging.exception("H2O shutdown failed")


def init():
    global _model, _manifest
    model_root = Path(os.environ["AZUREML_MODEL_DIR"]).resolve()
    manifest_path = _find_manifest(model_root)
    _manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    model_path = manifest_path.parent / _manifest["model_file"]
    model_format = _manifest.get("model_format")
    if model_format not in {"h2o_binary", "h2o_mojo"}:
        raise RuntimeError(f"Unsupported H2O model format: {model_format}")
    runtime_version = _manifest.get("runtime_h2o_version", _manifest["h2o_version"])
    if h2o.__version__ != runtime_version:
        raise RuntimeError(f"Expected runtime h2o=={runtime_version}, found {h2o.__version__}")
    if model_format == "h2o_binary" and runtime_version != _manifest["h2o_version"]:
        raise RuntimeError("Native H2O binaries require the producer H2O version")
    if _sha256(model_path) != _manifest["files"][model_path.name]:
        raise RuntimeError("Model checksum does not match the manifest")
    h2o.no_progress()
    h2o.init(
        ip="127.0.0.1",
        port=54321,
        start_h2o=True,
        nthreads=int(os.environ.get("H2O_NTHREADS", "3")),
        max_mem_size=os.environ.get("H2O_MAX_MEM_SIZE", "6G"),
        strict_version_check=True,
        bind_to_localhost=True,
        verbose=False,
        telemetry=False,
    )
    _model = (
        h2o.load_model(str(model_path))
        if model_format == "h2o_binary"
        else h2o.upload_mojo(str(model_path))
    )
    atexit.register(_shutdown_h2o)
    logging.info(
        "H2O model initialized: name=%s version=%s format=%s runtime=%s",
        _manifest["model_name"],
        _manifest["model_version"],
        model_format,
        runtime_version,
    )


def run(raw_data):
    try:
        frame = _parse_request(raw_data)
    except (ValueError, TypeError, KeyError, json.JSONDecodeError) as exc:
        return AMLResponse({"error": str(exc)}, 400, json_str=True)
    with _predict_lock:
        predictions = _predict(frame)
    return {
        "predictions": predictions,
        "model_name": _manifest["model_name"],
        "model_version": _manifest["model_version"],
        "model_format": _manifest["model_format"],
        "model_h2o_version": _manifest["h2o_version"],
        "runtime_h2o_version": _manifest.get("runtime_h2o_version", _manifest["h2o_version"]),
        "mojo_version": _manifest.get("mojo_version"),
    }
'''

ONLINE_SCORE_SOURCE = textwrap.dedent(ONLINE_SCORE_SOURCE).lstrip()
ast.parse(ONLINE_SCORE_SOURCE, filename=str(SCORE_PATH))
SCORE_PATH.write_text(ONLINE_SCORE_SOURCE, encoding="utf-8")
print(f"Generated and validated scoring script: {SCORE_PATH}")

Generated and validated scoring script: /home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop/outputs/generated/h2o_customer/online/score.py


## 3. Confirm the target workspace and switches

The next cell is read-only. It connects with your Azure CLI session and prints the workspace, resource group, deployment switch, and promotion switch.

For the review pass, both switches should print `False`.

In [4]:
credential = AzureCliCredential(
    tenant_id=os.getenv("AZURE_TENANT_ID") or None
)
ml_client = MLClient(
    credential,
    os.environ["AZURE_SUBSCRIPTION_ID"],
    os.environ["AZURE_RESOURCE_GROUP"],
    os.environ["AZUREML_WORKSPACE_NAME"],
)

workspace = ml_client.workspaces.get(os.environ["AZUREML_WORKSPACE_NAME"])
print(f"Target workspace: {workspace.name}")
print(f"Resource group: {os.environ['AZURE_RESOURCE_GROUP']}")
print(f"Deployment enabled: {DEPLOY}")
print(f"Traffic promotion enabled: {PROMOTE}")

Target workspace: mlwdevcc01
Resource group: rg-aml-ws-dev-cc-01
Deployment enabled: True
Traffic promotion enabled: True


## 4. Prepare the endpoint definition

The endpoint is the stable URL and identity boundary. If a user-assigned identity ID is provided, we attach it when the endpoint is first created. If it is blank, Azure creates a system-assigned identity.

Stop and correct `.env` if the identity is wrong. Reusing an existing endpoint does not replace its identity, because endpoint identity is immutable.

In [ ]:
identity = None
if IDENTITY_ID:
    identity = IdentityConfiguration(
        type=ManagedServiceIdentityType.USER_ASSIGNED,
        user_assigned_identities=[
            ManagedIdentityConfiguration(resource_id=IDENTITY_ID)
        ],
    )

endpoint_definition = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    auth_mode="aad_token",
    identity=identity,
    public_network_access=PUBLIC_NETWORK_ACCESS,
    description="Customer H2O model endpoint",
    tags={
        "workshop": "azureml-h2o-customer",
        "model": f"{MODEL_NAME}@latest"
    },
)

display(
    {
        "endpoint": ENDPOINT_NAME,
        "authentication": endpoint_definition.auth_mode,
        "identity": IDENTITY_ID or "system-assigned at endpoint creation",
        "public_network_access": PUBLIC_NETWORK_ACCESS,
    }
)

## 5. Prepare the deployment definition

The deployment binds four things together: the immutable model, the immutable environment, the scoring code, and the compute size. H2O memory and thread settings are passed to the scoring container as environment variables.

Review this summary before enabling deployment. Managed online endpoint instances incur cost while the deployment exists.

In [6]:
registered_model = ml_client.models.get(MODEL_NAME, label="latest")
registered_environment = ml_client.environments.get(
    ENVIRONMENT_NAME, label="latest"
)

deployment_definition = ManagedOnlineDeployment(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    model=registered_model,
    environment=registered_environment,
    code_configuration=CodeConfiguration(
        code=str(GENERATED_SCORE_DIR),
        scoring_script="score.py",
    ),
    instance_type=os.environ["AZUREML_ONLINE_INSTANCE_TYPE"],
    instance_count=1,
    app_insights_enabled=True,
    environment_variables={
        "WORKER_COUNT": "1",
        "H2O_NTHREADS": os.environ["H2O_NTHREADS"],
        "H2O_MAX_MEM_SIZE": os.environ["H2O_MAX_MEM_SIZE"],
    },
)

display(
    {
        "deployment": DEPLOYMENT_NAME,
        "model": f"{registered_model.name}:{registered_model.version}",
        "environment": f"{registered_environment.name}:{registered_environment.version}",
        "instance_type": deployment_definition.instance_type,
        "instance_count": deployment_definition.instance_count,
    }
)

{'deployment': 'blue',
 'model': 'workshop-h2o-customer-binary:1',
 'environment': 'workshop-h2o-customer-binary-environment:1',
 'instance_type': 'Standard_DS3_v2',
 'instance_count': 1}

## 6. Create the endpoint and deployment

This is the first cell that can write to Azure. It reuses the endpoint when it already exists and creates the deployment with the latest model and environment resolved above.

The deployment is addressed directly during testing. We do not assign traffic here. If provisioning does not finish in `Succeeded`, the notebook stops before logs, invocation, or promotion.

In [7]:
if DEPLOY:
    try:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
        print(f"Reusing endpoint: {endpoint.name}")
    except ResourceNotFoundError:
        endpoint = ml_client.online_endpoints.begin_create_or_update(
            endpoint_definition
        ).result()
        print(f"Created endpoint: {endpoint.name}")

    try:
        existing_deployment = ml_client.online_deployments.get(
            DEPLOYMENT_NAME,
            ENDPOINT_NAME,
        )
    except ResourceNotFoundError:
        existing_deployment = None
    if (
        existing_deployment is not None
        and existing_deployment.provisioning_state == "Failed"
    ):
        print(f"Deleting failed deployment: {DEPLOYMENT_NAME}")
        ml_client.online_deployments.begin_delete(
            DEPLOYMENT_NAME,
            ENDPOINT_NAME,
        ).result()

    deployment = ml_client.online_deployments.begin_create_or_update(
        deployment_definition
    ).result()

    if deployment.provisioning_state != "Succeeded":
        raise RuntimeError(
            f"Deployment state is {deployment.provisioning_state}"
        )

    print(
        f"Deployment ready: "
        f"{deployment.name} ({deployment.provisioning_state})"
    )
else:
    print(f"Prepared endpoint/deployment: {ENDPOINT_NAME}/{DEPLOYMENT_NAME}")
    print("Deployment is off. Set DEPLOY_H2O_ENDPOINT=true when you are ready.")

Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


Check: endpoint h2o-customer-binary-endpoint exists


Created endpoint: h2o-customer-binary-endpoint



Uploading online (0.01 MBs):   0%|                     | 0/5984 [00:00<?, ?it/s]

...

.

.

.

.

Deployment ready: blue (Succeeded)


## 7. Review the inference-server logs

A successful provisioning state tells us the Azure resource exists. The inference-server log tells us whether the generated scorer initialized cleanly and loaded the selected artifact with the configured runtime.

In [8]:
if DEPLOY:
    logs = ml_client.online_deployments.get_logs(
        DEPLOYMENT_NAME,
        ENDPOINT_NAME,
        100,
        container_type="inference-server",
    )
    print("\n".join(logs.splitlines()[-25:]))
else:
    print("Log retrieval skipped because deployment is off.")

2026-09-15 12:33:59,152 W [63] azmlinfsrv - Found extra keys in the config file that are not supported by the server.
Extra keys = ['AZUREML_ENTRY_SCRIPT', 'AZUREML_MODEL_DIR', 'HOSTNAME']
Initializing logger
2026-09-15 12:33:59,360 I [63] azmlinfsrv - Starting up app insights client
2026-09-15 12:34:00,416 I [63] azmlinfsrv.user_script - Found user script at /var/azureml-app/online/score.py
2026-09-15 12:34:00,416 I [63] azmlinfsrv.user_script - run() is not decorated. Server will invoke it with the input in JSON string.
2026-09-15 12:34:00,416 I [63] azmlinfsrv.user_script - Invoking user's init function
2026-09-15 12:34:05,304 I [63] azmlinfsrv.user_script - Users's init has completed successfully
2026-09-15 12:34:05,305 I [63] azmlinfsrv.swagger - Swaggers are prepared for the following versions: [2, 3, 3.1].
2026-09-15 12:34:05,305 I [63] azmlinfsrv - Scoring timeout is set to 3600000
2026-09-15 12:34:05,305 I [63] azmlinfsrv - Worker with pid 63 ready for serving traffic
2026-09-

## 8. Optionally build a golden request

When golden fixtures are present, the request uses the feature order stored in the manifest and is written under `outputs/generated`. Without golden fixtures, this step is skipped.

The expected predictions are loaded only for optional target-runtime parity.

In [9]:
request_path = None
golden_expected = None
if DEPLOY and GOLDEN_PROVIDED and RUN_GOLDEN_VALIDATION:
    golden_input = pd.read_csv(GOLDEN_INPUT_PATH)
    golden_expected = pd.read_csv(GOLDEN_EXPECTED_PATH)

    request = {
        "input_data": {
            "columns": manifest["features"],
            "data": golden_input[manifest["features"]].values.tolist(),
        }
    }
    request_path = GENERATED_SCORE_DIR / "golden_request.json"
    request_path.parent.mkdir(parents=True, exist_ok=True)
    request_path.write_text(
        json.dumps(request, indent=2),
        encoding="utf-8",
    )

    print(f"Request rows: {len(golden_input)}")
    print(f"Request file: {request_path}")
    display(request["input_data"]["columns"])
elif not DEPLOY:
    print("Golden request preparation skipped because deployment is off.")
elif GOLDEN_PROVIDED:
    print("Golden request preparation skipped by configuration.")
else:
    print("Golden request preparation skipped because no golden data was provided.")

Request rows: 20
Request file: /home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop/outputs/generated/h2o_customer/online/golden_request.json


['vendorID', 'passengerCount', 'tripDistance', 'paymentType', 'pickupHour']

## 9. Optionally invoke and compare predictions

When golden fixtures are present, we invoke the zero-traffic deployment directly and compare predictions. When they are absent, validation is skipped unless the manifest requires it.

A parity failure stops the notebook before the traffic cell.

In [10]:
def assert_prediction_parity(expected: pd.Series, actual: pd.Series) -> None:
    expected_numeric = pd.to_numeric(expected, errors="coerce")
    actual_numeric = pd.to_numeric(actual, errors="coerce")

    if expected_numeric.notna().all() and actual_numeric.notna().all():
        np.testing.assert_allclose(
            expected_numeric,
            actual_numeric,
            rtol=1e-6,
            atol=1e-6,
        )
        return

    pd.testing.assert_series_equal(
        expected.astype("string").reset_index(drop=True),
        actual.astype("string").reset_index(drop=True),
        check_names=False,
    )

RUNTIME_VALIDATION_PASSED = False
runtime_validation = {
    "status": "not_run" if not DEPLOY else "not_provided",
    "model": f"{registered_model.name}:{registered_model.version}",
    "environment": f"{registered_environment.name}:{registered_environment.version}",
    "model_format": manifest["model_format"],
    "runtime_h2o_version": manifest["runtime_h2o_version"],
}
if DEPLOY and GOLDEN_PROVIDED and RUN_GOLDEN_VALIDATION:
    raw_response = ml_client.online_endpoints.invoke(
        endpoint_name=ENDPOINT_NAME,
        deployment_name=DEPLOYMENT_NAME,
        request_file=str(request_path),
    )
    response = json.loads(raw_response)
    actual_predictions = pd.Series(response["predictions"])

    assert_prediction_parity(
        golden_expected["predict"],
        actual_predictions,
    )
    print(f"Cloud parity passed for {len(actual_predictions)} rows.")
    RUNTIME_VALIDATION_PASSED = True
    runtime_validation.update(
        {
            "status": "passed",
            "golden_rows": len(actual_predictions),
        }
    )
elif not DEPLOY:
    print("Cloud invocation skipped because deployment is off.")
elif GOLDEN_PROVIDED:
    runtime_validation["status"] = "skipped"
    print("Cloud parity skipped by configuration.")
else:
    print("Cloud parity skipped because no golden data was provided.")

if DEPLOY:
    validation_path = GENERATED_SCORE_DIR / "runtime_validation.json"
    validation_path.write_text(
        json.dumps(runtime_validation, indent=2) + "\n",
        encoding="utf-8",
    )
    print(f"Runtime validation evidence: {validation_path}")

if DEPLOY and REQUIRE_GOLDEN_VALIDATION and not RUNTIME_VALIDATION_PASSED:
    raise RuntimeError("Golden validation is required before this deployment can be promoted")

Cloud parity passed for 20 rows.
Runtime validation evidence: /home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop/outputs/generated/h2o_customer/online/runtime_validation.json


## 10. Promote traffic

Traffic promotion is a separate production decision. This cell changes traffic only when both `DEPLOY_H2O_ENDPOINT` and `PROMOTE_H2O_TRAFFIC` are `true`. Until then, the existing endpoint traffic stays exactly as it is.

After promotion, return `PROMOTE_H2O_TRAFFIC` to `false`.

In [11]:
if DEPLOY and PROMOTE:
    if REQUIRE_GOLDEN_VALIDATION and not RUNTIME_VALIDATION_PASSED:
        raise RuntimeError("Required golden validation has not passed")
    endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
    endpoint.traffic = {DEPLOYMENT_NAME: 100}
    ml_client.online_endpoints.begin_create_or_update(endpoint).result()
    print(f"Traffic promoted to {DEPLOYMENT_NAME}")
elif DEPLOY:
    print("Traffic remains unchanged because promotion is off.")
    print("Set PROMOTE_H2O_TRAFFIC=true only when you are ready to promote.")
else:
    print("Traffic promotion skipped because deployment is off.")

Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


Traffic promoted to blue


## Expected Result

The deployment reaches `Succeeded`, logs show MOJO import under the selected runtime, and direct invocation matches the customer's numeric predictions or class labels. Endpoint traffic changes only when the promotion switch is enabled.

Next: `05_submit_scoring_pipeline.ipynb`.